In [1]:
import time
from datetime import timedelta
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import segmentation_models_pytorch as smp
import json
import csv
from datetime import datetime
from patchify import patchify, unpatchify
from torchmetrics.functional.segmentation import mean_iou
from torchmetrics.functional.classification import multiclass_accuracy
# import wandb  # Opcional: pip install wandb

/home/calebe/geo-projects-car/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os

# Adiciona o diretório pai (..) ao path do sistema
sys.path.append(os.path.abspath('..'))

# Agora você pode importar seu arquivo normalmente
from train_code import PatchifyInference

In [3]:
def load_best_model(checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device)

    cfg = ckpt.get("config", {})
    model = smp.Unet(
        encoder_name=cfg.get("encoder_name", "resnet50"),
        encoder_weights=None,  # evita baixar pesos; vamos carregar do checkpoint
        in_channels=3,
        classes=cfg.get("num_classes", 2),
        activation=cfg.get("activation", None),
    ).to(device)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, cfg

In [4]:
def run_inference_from_checkpoint(checkpoint_path, image_paths, save_dir,
                                  image_size=2048, patch_size=256):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model, cfg = load_best_model(checkpoint_path, device)

    # Transform consistente com o seu val_transform atual
    infer_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    infer = PatchifyInference(
        model=model,
        device=device,
        image_size=cfg.get("image_size", image_size),
        patch_size=cfg.get("patch_size", patch_size),
        num_classes=cfg.get("num_classes", 2),
    )

    os.makedirs(save_dir, exist_ok=True)
    for img_path in image_paths:
        pred_mask = infer.predict_image(img_path, transform=infer_transform)
        out_path = os.path.join(save_dir, "pred_" + os.path.basename(img_path))
        Image.fromarray(pred_mask.astype("uint8")).save(out_path)

# Exemplo de chamada:
run_inference_from_checkpoint(
    checkpoint_path="/home/calebe/geo-projects-car/experiments/trying-use-class-weights-based_20260303_150328/best_model.pth",
    image_paths=[
        "/data/integracar/satellite_sample_2058/ES-3201605-6D3B7D2F0DA54E37A1DB8D30B9D72C9E.tif", 
        "/data/integracar/satellite_sample_2058/ES-3201803-4EB246F91D14462AACC60E101C6310C8.tif",
        "/data/integracar/satellite_sample_2058/ES-3204906-FEC6452DE4354398BB52137330D93B31.tif"],
    save_dir="preds_out"
)


In [5]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def plot_overlay_comparison(
    image_path: str,
    pred_mask,                 # np.ndarray (H,W) ou path para máscara
    gt_mask_path: str = None,  # opcional
    save_path: str = None,
    alpha: float = 0.6,
    class_colors=None,         # ex.: ['black','green'] ou ['black','red','green']
    titles=None
):
    original = Image.open(image_path).convert("RGB")

    if isinstance(pred_mask, str):
        pred = np.array(Image.open(pred_mask))
    else:
        pred = np.asarray(pred_mask)

    gt = None
    if gt_mask_path is not None:
        gt = np.array(Image.open(gt_mask_path))

    if class_colors is None:
        class_colors = ["gray", "green"]
    cmap = ListedColormap(class_colors)

    if titles is None:
        titles = {
            "gt": "Input + Ground Truth Overlay",
            "pred": "Article Based"
        }
    if gt is not None:
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(original)
        axes[0].imshow(gt, cmap=cmap, alpha=alpha, interpolation="none")
        axes[0].set_title(titles["gt"], fontsize=12, fontweight="bold")
        axes[0].axis("off")

        axes[1].imshow(original)
        axes[1].imshow(pred, cmap=cmap, alpha=alpha, interpolation="none")
        axes[1].set_title(titles["pred"], fontsize=12, fontweight="bold")
        axes[1].axis("off")
    else:
        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        ax.imshow(original)
        ax.imshow(pred, cmap=cmap, alpha=alpha, interpolation="none")
        ax.set_title(titles["pred"], fontsize=12, fontweight="bold")
        ax.axis("off")

    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


In [6]:
plot_overlay_comparison(
    image_path="/data/integracar/satellite_sample_2058/ES-3204906-FEC6452DE4354398BB52137330D93B31.tif",
    pred_mask="/home/calebe/geo-projects-car/notebooks/preds_out/pred_ES-3204906-FEC6452DE4354398BB52137330D93B31.tif",
    gt_mask_path="/data/integracar/amostras_car_mask_2058/ES-3204906-FEC6452DE4354398BB52137330D93B31.tif",
    save_path="overlay_img_002.png",
    alpha=0.6,
    class_colors=["gray", "green"]
)